# 02 — Text Analytics Pipeline: F0 → F5

**Clay Harris (jbm2rt@virginia.edu) / Text as Data / 2026-05-07**

This notebook runs the full text analytics pipeline on the papal encyclicals corpus.
Each stage transforms the data and adds annotations, culminating in three unsupervised models.

| Stage | Input → Output | Key Tables |
|-------|----------------|------------|
| F1 | Raw text → paragraphs | F1 corpus |
| F2 | Paragraphs → tokens | LIBRARY, TOKEN, VOCAB |
| F3 | Tokens → annotations | TOKEN+, VOCAB+, DOC_SENTIMENT |
| F4 | Annotations → weights | TFIDF_DTM |
| F5 | Weights → models | DOC_PCA, DOC_TOPICS, EMBEDDINGS |

In [1]:
# ── Configuration ──────────────────────────────────────────────────────
ENGLISH_ONLY  = True
N_COMPONENTS  = 10
N_TOPICS      = 10
W2V_DIM       = 100
FORCE_RERUN   = False   # Set True to recompute all stages from scratch

In [2]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))

from src.pipeline import (
    build_f1_corpus, build_f2_tables, build_f3_annotations,
    build_f4_tfidf, build_f5_models, save_tables,
    cache_exists, save_cache, load_cache,
    PROCESSED_DIR, CACHE_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

Imports OK


## F1: Machine Learning Corpus Format

Reads raw `.txt` files from `data/raw/` and the metadata index `encyclicals_index.json`.
Each document is split into paragraphs (the minimum discursive unit), producing one row per
paragraph with columns `doc_id`, `pope`, `title`, `year`, `para_num`, and `para_text`.
Setting `ENGLISH_ONLY=True` restricts processing to documents detected as English by the scraper.

In [3]:
%%time
if not FORCE_RERUN and cache_exists('f1'):
    corpus = load_cache('f1')
else:
    corpus = build_f1_corpus(english_only=ENGLISH_ONLY)
    save_cache('f1', corpus)

2026-05-07 19:11:32,734 [INFO] Loading cache: f1.pkl


CPU times: user 104 ms, sys: 50.9 ms, total: 155 ms
Wall time: 161 ms


In [4]:
print(f'CORPUS docs:  {corpus["doc_id"].nunique():,}')
print(f'CORPUS paras: {len(corpus):,}')
corpus.head()

CORPUS docs:  17,352
CORPUS paras: 406,631


,doc_id,pope,title,year,para_num,para_text
0,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,0,INTRODUCTION
1,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,1,This council was summoned by pope Julius II by...
2,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,2,There were twelve sessions. The first five of ...
3,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,3,"All the decrees of this council, at which the ..."
4,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,4,The decisions on the reform of the curia produ...


## F2: STADM Tables — LIBRARY, TOKEN, VOCAB

Applies NLTK sentence and word tokenization to each paragraph, producing the three core
Standard Text Analytic Data Model tables:

- **LIBRARY** — one row per document with metadata and aggregate statistics
- **TOKEN** — one row per token, indexed by the OHCO hierarchy `(doc_id, para_num, sent_num, token_num)`
- **VOCAB** — one row per unique term, with raw frequency and document frequency

In [5]:
%%time
if not FORCE_RERUN and cache_exists('f2'):
    LIBRARY, TOKEN, VOCAB = load_cache('f2')
else:
    LIBRARY, TOKEN, VOCAB = build_f2_tables(corpus)
    save_cache('f2', (LIBRARY, TOKEN, VOCAB))

2026-05-07 19:11:32,969 [INFO] Loading cache: f2.pkl


CPU times: user 2.89 s, sys: 1.46 s, total: 4.35 s
Wall time: 5.26 s


In [6]:
print(f'LIBRARY: {len(LIBRARY):,} documents')
print(f'TOKEN:   {len(TOKEN):,} tokens')
print(f'VOCAB:   {len(VOCAB):,} unique terms')
LIBRARY.head()

LIBRARY: 17,352 documents
TOKEN:   28,425,031 tokens
VOCAB:   219,533 unique terms


,pope,title,year,n_paragraphs,n_chars,n_tokens
doc_id,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,196,211930,40440
church_councils__the_first_general_council_of_constantinople__381,Church Councils,"The First General Council of Constantinople, 381",,51,22891,4427
church_councils__the_first_general_council_of_lyons__1245,Church Councils,"The First General Council of Lyons, 1245",1245,94,77512,15108
church_councils__the_first_general_council_of_nicaea__325,Church Councils,"The First General Council of Nicaea, 325",,20,6097,1169
church_councils__the_first_general_council_of_the_lateran__1123,Church Councils,"The First General Council of the Lateran, 1123",,37,16643,3526


In [7]:
TOKEN.head(10)

,doc_id,para_num,sent_num,token_num,token_str,term_str
token_id,,,,,,
0,church_councils__the_fifth_general_council_of_...,0,0,0,INTRODUCTION,introduction
1,church_councils__the_fifth_general_council_of_...,1,0,0,This,this
2,church_councils__the_fifth_general_council_of_...,1,0,1,council,council
3,church_councils__the_fifth_general_council_of_...,1,0,2,was,was
4,church_councils__the_fifth_general_council_of_...,1,0,3,summoned,summoned
5,church_councils__the_fifth_general_council_of_...,1,0,4,by,by
6,church_councils__the_fifth_general_council_of_...,1,0,5,pope,pope
7,church_councils__the_fifth_general_council_of_...,1,0,6,Julius,julius
8,church_councils__the_fifth_general_council_of_...,1,0,7,II,ii


In [8]:
VOCAB.head(20)

,n,df,idf
term_str,,,
the,1908439,17337,0.000865
",",1610794,17342,0.000576
of,1247569,17337,0.000865
.,944840,17190,0.009380
and,898628,17293,0.003406
to,843205,17317,0.002019
in,588292,17294,0.003348
is,341956,16667,0.040277
a,341438,17105,0.014337


## F3: NLP Annotations — POS, Lemma, Stopwords, Sentiment

Enriches TOKEN and VOCAB with linguistic annotations:

- **POS tags** via NLTK averaged perceptron tagger (Penn Treebank tagset)
- **Lemmas** via NLTK WordNetLemmatizer (using POS context to disambiguate)
- **Stopword flags** from NLTK English stopword list
- **VADER sentiment** on each vocabulary term (neg/neu/pos/compound)
- **DOC_SENTIMENT** — aggregate VADER compound/pos/neg/neu scores per document

This is the most compute-intensive stage. The pickle checkpoint makes reruns near-instant.

In [9]:
%%time
if not FORCE_RERUN and cache_exists('f3'):
    LIBRARY, TOKEN, VOCAB, DOC_SENTIMENT = load_cache('f3')
else:
    LIBRARY, TOKEN, VOCAB, DOC_SENTIMENT = build_f3_annotations(TOKEN, VOCAB, LIBRARY)
    save_cache('f3', (LIBRARY, TOKEN, VOCAB, DOC_SENTIMENT))

2026-05-07 19:11:38,289 [INFO] Building F3 NLP annotations...
2026-05-07 19:11:40,995 [INFO]   POS tagging...
POS tagging: 100%|██████████| 1169671/1169671 [07:53<00:00, 2469.92it/s]
2026-05-07 19:19:54,411 [INFO]   Lemmatizing...
Lemmatizing: 100%|██████████| 28425031/28425031 [08:03<00:00, 58841.67it/s]
2026-05-07 19:28:03,444 [INFO]   Updating VOCAB...
2026-05-07 19:28:27,411 [INFO]   Computing VADER sentiment for vocab...
VADER vocab: 100%|██████████| 219533/219533 [00:02<00:00, 100698.58it/s]
2026-05-07 19:28:29,683 [INFO]   Computing document-level sentiment...
Doc sentiment: 100%|██████████| 17352/17352 [5:04:11<00:00,  1.05s/it]  
2026-05-08 00:32:41,456 [INFO]   F3 annotations complete
2026-05-08 00:33:03,657 [INFO] Cache saved: f3.pkl


CPU times: user 5h 17min, sys: 2min 22s, total: 5h 19min 23s
Wall time: 5h 21min 25s


In [10]:
print(f'TOKEN cols:    {list(TOKEN.columns)}')
print(f'VOCAB cols:    {list(VOCAB.columns)}')
print(f'DOC_SENTIMENT: {len(DOC_SENTIMENT):,} documents')
TOKEN.head(10)

TOKEN cols:    ['doc_id', 'para_num', 'sent_num', 'token_num', 'token_str', 'term_str', 'pos', 'lemma', 'is_stop', 'is_alpha']
VOCAB cols:    ['n', 'df', 'idf', 'pos', 'lemma', 'is_stop', 'vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']
DOC_SENTIMENT: 17,352 documents


,doc_id,para_num,sent_num,token_num,token_str,term_str,pos,lemma,is_stop,is_alpha
token_id,,,,,,,,,,
0,church_councils__the_fifth_general_council_of_...,0,0,0,INTRODUCTION,introduction,NN,introduction,False,True
1,church_councils__the_fifth_general_council_of_...,1,0,0,This,this,DT,this,True,True
2,church_councils__the_fifth_general_council_of_...,1,0,1,council,council,NN,council,False,True
3,church_councils__the_fifth_general_council_of_...,1,0,2,was,was,VBD,be,True,True
4,church_councils__the_fifth_general_council_of_...,1,0,3,summoned,summoned,VBN,summon,False,True
5,church_councils__the_fifth_general_council_of_...,1,0,4,by,by,IN,by,True,True
6,church_councils__the_fifth_general_council_of_...,1,0,5,pope,pope,NN,pope,False,True
7,church_councils__the_fifth_general_council_of_...,1,0,6,Julius,julius,NNP,julius,False,True
8,church_councils__the_fifth_general_council_of_...,1,0,7,II,ii,NNP,ii,False,True


In [11]:
VOCAB.head(10)

,n,df,idf,pos,lemma,is_stop,vader_neg,vader_neu,vader_pos,vader_compound
term_str,,,,,,,,,,
the,1908439,17337,0.000865,DT,the,True,0.0,1.0,0.0,0.0
",",1610794,17342,0.000576,",",",",False,0.0,0.0,0.0,0.0
of,1247569,17337,0.000865,IN,of,True,0.0,1.0,0.0,0.0
.,944840,17190,0.009380,.,.,False,0.0,0.0,0.0,0.0
and,898628,17293,0.003406,CC,and,True,0.0,1.0,0.0,0.0
to,843205,17317,0.002019,TO,to,True,0.0,1.0,0.0,0.0
in,588292,17294,0.003348,IN,in,True,0.0,1.0,0.0,0.0
is,341956,16667,0.040277,VBZ,be,True,0.0,1.0,0.0,0.0
a,341438,17105,0.014337,DT,a,True,0.0,0.0,0.0,0.0


In [12]:
print('DOC_SENTIMENT (document-level VADER scores):')
DOC_SENTIMENT.head(10)

DOC_SENTIMENT (document-level VADER scores):


,vader_neg,vader_neu,vader_pos,vader_compound
doc_id,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,0.036,0.865,0.099,0.9996
church_councils__the_first_general_council_of_constantinople__381,0.073,0.800,0.126,0.9997
church_councils__the_first_general_council_of_lyons__1245,0.088,0.772,0.141,0.9998
church_councils__the_first_general_council_of_nicaea__325,0.043,0.781,0.175,0.9995
church_councils__the_first_general_council_of_the_lateran__1123,0.068,0.861,0.071,-0.6946
church_councils__the_first_general_council_of_the_vatican__1869_70,0.032,0.786,0.183,0.9997
church_councils__the_fourth_general_council_of_constantinople__869_,0.096,0.736,0.168,0.9999
church_councils__the_fourth_general_council_of_the_lateran__1215,0.068,0.793,0.139,0.9999
church_councils__the_general_council_of_basel_ferrara_florence__143,0.040,0.840,0.120,0.9999


## F4: TFIDF Vectorization

Computes normalized term frequency (TF), inverse document frequency (IDF), and TFIDF for
every (document, term) pair. Produces the **TFIDF_DTM** document-term matrix, which is the
primary input to the unsupervised models in F5.

Only meaningful terms (alphabetic, non-stop, `df > 1`) are included as DTM columns
to control dimensionality.

In [13]:
%%time
if not FORCE_RERUN and cache_exists('f4'):
    LIBRARY, TOKEN, VOCAB, TFIDF_DTM = load_cache('f4')
else:
    LIBRARY, TOKEN, VOCAB, TFIDF_DTM = build_f4_tfidf(TOKEN, VOCAB, LIBRARY)
    save_cache('f4', (LIBRARY, TOKEN, VOCAB, TFIDF_DTM))

2026-05-08 00:33:03,783 [INFO] Loading cache: f4.pkl


CPU times: user 561 ms, sys: 145 ms, total: 706 ms
Wall time: 753 ms


In [14]:
print(f'TFIDF_DTM shape: {TFIDF_DTM.shape}  (docs x terms)')
print(f'\nTop TFIDF terms — first 5 documents:')
for doc_id in TFIDF_DTM.index[:5]:
    top = TFIDF_DTM.loc[doc_id].nlargest(5)
    print(f'  {doc_id[:50]}: {", ".join(top.index)}')
TFIDF_DTM.head()

TFIDF_DTM shape: (553, 29758)  (docs x terms)

Top TFIDF terms — first 5 documents:
  church_councils__the_fifth_general_council_of_the_: benefices, council, cardinals, approval, prelates
  church_councils__the_first_general_council_of_cons: constantinople, contents, arians, nicene, synod
  church_councils__the_first_general_council_of_lyon: excommunication, prelates, coll, plaintiff, empire
  church_councils__the_first_general_council_of_nica: alexander, synod, eusebius, meletius, tanner
  church_councils__the_first_general_council_of_the_: canons, alpha, msi, b, forbid


term_str,aa,aand,aargau,aaron,aas,ab,abandon,abandoned,abandoning,abandonment,...,âme,âmes,è,église,émile,étienne,évêques,être,ô,über
doc_id,,,,,,,,,,,,,,,,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,0.000000,0.0,0.0,0.0,0.0,0.0,0.000127,0.000041,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
church_councils__the_first_general_council_of_constantinople__381,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
church_councils__the_first_general_council_of_lyons__1245,0.000299,0.0,0.0,0.0,0.0,0.0,0.000000,0.000109,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
church_councils__the_first_general_council_of_nicaea__325,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.001413,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
church_councils__the_first_general_council_of_the_lateran__1123,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## F5: Unsupervised Models — PCA, LDA, Word2Vec

Fits three complementary unsupervised models:

1. **PCA** — principal component analysis on the TFIDF matrix. Reveals the primary axes of
   topical variation. Outputs `DOC_PCA`, `LOADINGS`, and `explained_variance`.

2. **LDA** — Latent Dirichlet Allocation topic model. Finds soft clusters of co-occurring terms
   and assigns each document a mixture of topics. Outputs `DOC_TOPICS` and `TOPIC_TERMS`.

3. **Word2Vec** — neural term embeddings trained on lemmatized sentences. Captures semantic
   similarity in a dense vector space. Outputs `EMBEDDINGS`.

In [15]:
%%time
if not FORCE_RERUN and cache_exists('f5'):
    f5_results = load_cache('f5')
else:
    f5_results = build_f5_models(
        LIBRARY, TOKEN, VOCAB, TFIDF_DTM,
        n_components=N_COMPONENTS,
        n_topics=N_TOPICS,
        w2v_dim=W2V_DIM,
    )
    save_cache('f5', f5_results)

2026-05-08 00:33:04,576 [INFO] Loading cache: f5.pkl


CPU times: user 132 ms, sys: 74.5 ms, total: 206 ms
Wall time: 606 ms


In [16]:
DOC_PCA = f5_results['DOC_PCA']
LOADINGS = f5_results['LOADINGS']
explained_variance = f5_results['explained_variance']

print(f'DOC_PCA shape: {DOC_PCA.shape}')
print(f'Explained variance (cumulative): {explained_variance.cumsum().round(3).tolist()}')
DOC_PCA.head()

DOC_PCA shape: (553, 10)
Explained variance (cumulative): [0.019, 0.029, 0.037, 0.044, 0.051, 0.057, 0.063, 0.069, 0.074, 0.08]


,PC0,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9
doc_id,,,,,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,-9.255367,-1.034517,-6.970709,3.007812,3.809368,-1.166608,-1.399170,-3.038342,0.971361,-0.861139
church_councils__the_first_general_council_of_constantinople__381,-8.093166,-1.397936,-5.698969,0.964374,2.390057,-0.489956,-0.428185,-2.263125,0.161612,0.225377
church_councils__the_first_general_council_of_lyons__1245,-9.138159,-1.170361,-9.708592,2.879002,5.280274,-1.747786,-2.177641,-3.729213,1.911872,-1.619041
church_councils__the_first_general_council_of_nicaea__325,-5.330178,-0.073353,-7.775234,0.729082,0.687644,0.223570,-0.380294,-1.278364,0.374785,0.191952
church_councils__the_first_general_council_of_the_lateran__1123,-4.982770,2.161922,-10.429436,3.260040,2.061778,0.078106,0.386798,-5.318203,0.315758,-0.232417


In [17]:
DOC_TOPICS = f5_results['DOC_TOPICS']
TOPIC_TERMS = f5_results['TOPIC_TERMS']

print(f'DOC_TOPICS shape: {DOC_TOPICS.shape}')
print('\nTop terms per topic:')
for col in TOPIC_TERMS.columns:
    top = TOPIC_TERMS[col].nlargest(10).index.tolist()
    print(f'  {col}: {", ".join(top)}')
DOC_TOPICS.head()

DOC_TOPICS shape: (553, 10)

Top terms per topic:
  topic_0: church, life, christ, god, christian, people, cf, faith, spirit, one
  topic_1: church, catholic, bishop, see, apostolic, rite, holy, unity, roman, faith
  topic_2: god, christ, love, life, man, one, word, church, jesus, spirit
  topic_3: science, research, method, moral, scientific, psychology, principle, personality, psychic, patient
  topic_4: et, de, la, le, ad, est, ed, qui, non, que
  topic_5: di, che, la, il, per, non, si, del, della, le
  topic_6: church, god, may, men, great, christian, catholic, divine, one, christ
  topic_7: human, life, man, good, social, must, one, right, people, world
  topic_8: council, holy, church, synod, god, say, one, father, christ, bishop
  topic_9: church, order, may, cardinal, decree, council, make, person, say, holy


,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9
doc_id,,,,,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,0.000006,0.000006,0.000006,0.000006,0.000563,0.000006,0.183826,0.015108,0.088253,0.712218
church_councils__the_first_general_council_of_constantinople__381,0.010225,0.119176,0.044489,0.000063,0.000063,0.000063,0.101212,0.031235,0.537014,0.156461
church_councils__the_first_general_council_of_lyons__1245,0.000018,0.000018,0.000018,0.000018,0.000018,0.000018,0.193195,0.000018,0.076026,0.730652
church_councils__the_first_general_council_of_nicaea__325,0.000248,0.192495,0.079068,0.000248,0.000248,0.000248,0.094094,0.019953,0.338274,0.275125
church_councils__the_first_general_council_of_the_lateran__1123,0.015839,0.139353,0.000095,0.000095,0.022358,0.000095,0.000095,0.000095,0.069801,0.752174


In [18]:
EMBEDDINGS = f5_results['EMBEDDINGS']
print(f'EMBEDDINGS shape: {EMBEDDINGS.shape}  (terms x dimensions)')
EMBEDDINGS.head()

EMBEDDINGS shape: (15832, 100)  (terms x dimensions)


,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,...,w2v_90,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99
term_str,,,,,,,,,,,,,,,,,,,,,
the,-1.209627,-1.594323,0.276932,0.095530,2.675643,0.388169,0.001174,-1.297609,-0.735487,-0.283592,...,-0.238713,-0.752752,-1.167086,1.555164,-0.166393,0.148137,2.136510,0.160282,1.059817,1.048951
of,-0.655693,-1.483039,0.135197,-0.651447,1.877446,1.298649,-0.700949,-0.230431,0.591355,-0.120585,...,0.965525,-0.533814,-0.658913,1.172513,1.327514,-0.198872,1.199090,0.030504,0.336472,1.238208
and,-0.721130,-1.745455,0.938710,1.432758,2.814915,-0.257602,0.603853,-0.771953,-1.316661,-0.304972,...,-0.098223,-0.501793,-0.162042,1.108662,0.187759,0.071795,0.428814,-1.322136,0.548598,-0.074046
be,-0.848593,-2.183878,-0.128740,0.114353,1.957285,1.050105,0.756085,-0.515936,-1.683642,0.583978,...,-0.346824,-1.789784,-1.848060,1.151399,-0.679529,0.455548,0.252272,-2.386683,-0.746463,0.594036
to,-0.584541,-0.326784,0.816523,-0.153787,2.021468,1.337074,1.325366,-1.955945,0.184215,-0.970704,...,1.360892,-1.129416,0.336648,2.240060,0.369132,0.537171,1.236309,-2.236352,0.475438,1.540357


## Save All Deliverables

Write all tables to `data/processed/*.csv` and print a manifest of output files.

In [19]:
save_tables(LIBRARY, TOKEN, VOCAB, TFIDF_DTM, DOC_SENTIMENT=DOC_SENTIMENT,
            f5_results=f5_results)

print('\n── Output manifest ───────────────────────────────────────────────')
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} {size_kb:>8.0f} KB')
print('────────────────────────────────────────────────────────────────')
print('Pipeline complete!')

2026-05-08 00:33:05,231 [INFO] Saving tables to CSV...
2026-05-08 00:33:26,492 [INFO] All tables saved to /Users/queclay/Documents/MSDS/DS5001/encyclicals/data/processed/



── Output manifest ───────────────────────────────────────────────
  DOC_PCA.csv                              126 KB
  DOC_SENTIMENT.csv                       2013 KB
  DOC_TOPICS.csv                           136 KB
  EMBEDDINGS.csv                         18354 KB
  LIBRARY.csv                               64 KB
  LOADINGS.csv                            6787 KB
  POPE_COVERAGE.csv                          3 KB
  TFIDF_DTM.csv                          75662 KB
  TOKEN.csv                             444273 KB
  TOPIC_TERMS.csv                          979 KB
  VOCAB.csv                               7731 KB
  dead_link_replacements.csv                10 KB
  explained_variance.csv                     0 KB
  vatican_pope_destinations.csv             41 KB
────────────────────────────────────────────────────────────────
Pipeline complete!
